# SAM-AI Sovereign Agent: Real Base-Model Smoke Test (Zero-Shot Baseline)

**Objective:** Verify that the sovereign SAM-AI agent (`sam_ai.agent`) loads authentic 4-bit base model weights (`DeepSeek-R1-Distill-Qwen-14B-bnb-4bit`) via `HuggingFaceModelRunner`, connects to the official ARC-3 `Arcade` environment interface, and executes a strictly budgeted 5-step interactive loop.

> **Hard Rules:**
> 1. **Zero competition submission:** Does not create `submission.parquet` or invoke submission APIs.
> 2. **Base-Model Only:** Does not mount, load, or evaluate the SAM-AI adapter. All results labeled `REAL_MODEL_ZERO_SHOT_BASE`.
> 3. **No mock fallback:** If weights, GPU, or environment data are absent, halts cleanly with `[BLOCKED]`.
> 4. **Small compute footprint:** Bounded to 5 environment steps total (< 2 minutes GPU runtime).
> 5. **Architectural Parity:** Uses identical `HuggingFaceModelRunner` and telemetry schema as the local `scripts/run_real_model_smoke_test.py`.


In [ ]:
"""Kaggle input path discovery module for SAM-AI smoke tests.

Standard-library only. Discovers input paths by content rather than brittle
hard-coded path assumptions. Finds:
- sam_ai_parent: directory containing the sam_ai package directory
- environment_files: directory containing ARC-AGI-3 environment metadata and game files
- base_model: directory containing model config.json and weights
"""

import os
import sys
import json
import tarfile
import zipfile
import re
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple


def print_layout(root: Path, max_entries: int = 150) -> str:
    """Scan and format the directory layout for diagnostics."""
    lines = [f"=== INPUT LAYOUT ({root}) ==="]
    if not root.exists():
        lines.append(f"  <Directory does not exist: {root}>")
        text = "\n".join(lines)
        print(text)
        return text

    count = 0
    for current, dirs, files in os.walk(root):
        rel = os.path.relpath(current, root)
        if rel != ".":
            lines.append(f"  [DIR]  {rel}")
            count += 1
        for f in files:
            rel_file = os.path.normpath(os.path.join(rel if rel != "." else "", f))
            lines.append(f"  [FILE] {rel_file}")
            count += 1
            if count >= max_entries:
                lines.append(f"  ... [truncated at {max_entries} entries]")
                break
        if count >= max_entries:
            break

    text = "\n".join(lines)
    print(text)
    return text


def _is_valid_env_dir(d: Path) -> bool:
    """Check if directory contains at least one metadata.json with game files."""
    if not d.is_dir():
        return False
    metas = list(d.rglob("metadata.json"))
    if not metas:
        return False
    for m in metas:
        try:
            with open(m, "r", encoding="utf-8") as f:
                data = json.load(f)
            game_id = data.get("game_id")
            class_name = data.get("class_name") or (game_id[0].upper() + game_id[1:] if game_id else "Game")
            candidates = [
                m.parent / f"{class_name.lower()}.py",
                m.parent / f"{class_name}.py",
                m.parent / f"{game_id}.py",
            ]
            if any(c.is_file() for c in candidates):
                return True
        except Exception:
            continue
    return False


def _is_valid_model_dir(d: Path) -> bool:
    """Check if directory contains a Qwen/DeepSeek config.json and weights."""
    if not d.is_dir():
        return False
    cfg_file = d / "config.json"
    if not cfg_file.is_file():
        return False
    try:
        with open(cfg_file, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        model_type = cfg.get("model_type", "").lower()
        if model_type not in ("qwen2", "deepseek_v2", "deepseek_v3"):
            return False
        weights = list(d.glob("*.safetensors")) + list(d.glob("*.bin"))
        return len(weights) > 0
    except Exception:
        return False


def discover_inputs(
    input_dir: str | Path = "/kaggle/input",
    working_dir: str | Path = "/kaggle/working",
) -> Dict[str, Path]:
    """Discover paths to sam_ai, environment_files, and base_model by content.

    Prints the /kaggle/input layout first. Raises FileNotFoundError with layout
    if any required prerequisite is missing.
    """
    input_root = Path(input_dir).resolve()
    working_root = Path(working_dir).resolve()
    working_root.mkdir(parents=True, exist_ok=True)

    # 1. Print full input layout for diagnostics
    layout_str = print_layout(input_root)

    sam_ai_parent: Optional[Path] = None
    env_dir: Optional[Path] = None
    base_model: Optional[Path] = None

    if input_root.exists():
        # A. Find sam_ai package directory
        for current, dirs, files in os.walk(input_root):
            depth = len(Path(current).relative_to(input_root).parts)
            if depth > 6:
                dirs.clear()
                continue
            if "sam_ai" in dirs:
                candidate = Path(current) / "sam_ai"
                if (candidate / "__init__.py").is_file():
                    sam_ai_parent = Path(current)
                    break

        # A2. Archive fallback if sam_ai directory not found
        if not sam_ai_parent:
            for current, dirs, files in os.walk(input_root):
                for f in files:
                    if ("sam_ai" in f.lower() or "source" in f.lower()) and f.endswith((".tar.gz", ".tgz")):
                        arch_path = Path(current) / f
                        extract_dest = working_root / "sam_ai_extracted"
                        extract_dest.mkdir(parents=True, exist_ok=True)
                        with tarfile.open(arch_path, "r:*") as tar:
                            for m in tar.getmembers():
                                if m.issym() or m.islnk() or ".." in m.name:
                                    raise RuntimeError(f"Unsafe archive member: {m.name}")
                            tar.extractall(extract_dest)
                        for c_root, c_dirs, _ in os.walk(extract_dest):
                            if "sam_ai" in c_dirs and (Path(c_root) / "sam_ai" / "__init__.py").is_file():
                                sam_ai_parent = Path(c_root)
                                break
                        if sam_ai_parent:
                            break
                if sam_ai_parent:
                    break

        # B. Find environment_files
        # Prioritize explicit environment_files directories
        for current, dirs, files in os.walk(input_root):
            depth = len(Path(current).relative_to(input_root).parts)
            if depth > 6:
                dirs.clear()
                continue
            if Path(current).name == "environment_files" and _is_valid_env_dir(Path(current)):
                env_dir = Path(current)
                break
        if not env_dir:
            for current, dirs, files in os.walk(input_root):
                p = Path(current)
                if _is_valid_env_dir(p):
                    env_dir = p
                    break

        # C. Find base_model directory
        for current, dirs, files in os.walk(input_root):
            depth = len(Path(current).relative_to(input_root).parts)
            if depth > 6:
                dirs.clear()
                continue
            p = Path(current)
            if _is_valid_model_dir(p):
                base_model = p
                break

    # Validate findings
    missing = []
    if not sam_ai_parent:
        missing.append("sam_ai package (directory containing sam_ai/__init__.py)")
    if not env_dir:
        missing.append("ARC-3 environment_files (directory with metadata.json and game .py)")
    if not base_model:
        missing.append("Base model directory (directory with config.json and weights)")

    if missing:
        err_msg = (
            f"Failed to discover required Kaggle input paths: {', '.join(missing)}\n\n"
            f"{layout_str}"
        )
        raise FileNotFoundError(err_msg)

    print(f"[OK] Discovered sam_ai_parent:   {sam_ai_parent}")
    print(f"[OK] Discovered environment_dir: {env_dir}")
    print(f"[OK] Discovered base_model:      {base_model}")

    return {
        "sam_ai_parent": sam_ai_parent,
        "environment_files": env_dir,
        "base_model": base_model,
        "env_dir": env_dir,  # alias for backwards compatibility
    }


if __name__ == "__main__":
    try:
        inp = sys.argv[1] if len(sys.argv) > 1 else "/kaggle/input"
        res = discover_inputs(input_dir=inp)
        print("Discovery succeeded:", res)
    except Exception as e:
        print("Discovery failed:", e)
        sys.exit(1)


In [ ]:
# ==============================================================================
# 1. Environment, Dependencies & Static Precondition Audit
# ==============================================================================
import os
import sys
import time
import json
import subprocess
from pathlib import Path

print('=' * 75)
print('  SAM-AI REAL BASE-MODEL SMOKE TEST: STATIC PRECONDITION AUDIT')
print('=' * 75)

# 0. Discover input paths by content
paths = discover_inputs()
sam_ai_parent = str(paths['sam_ai_parent'])
if sam_ai_parent not in sys.path:
    sys.path.insert(0, sam_ai_parent)

base_model_path = str(paths['base_model'])
ENV_DATA_DIR = str(paths['environment_files'])

print(f'[OK] sam_ai parent resolved:   {sam_ai_parent}')
print(f'[OK] Base Model resolved:      {base_model_path}')
print(f'[OK] Environment dir resolved: {ENV_DATA_DIR}')

# 1. Check and Install Official arc-agi / arcengine Wheels Offline
try:
    import arc_agi
    import arcengine
    print('[OK] arc-agi and arcengine packages are available.')
except ImportError:
    wheel_dirs = [
        Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
        Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
    ]
    installed = False
    for wd in wheel_dirs:
        if wd.exists() and list(wd.glob('*.whl')):
            print(f'[*] Installing offline wheels from {wd}...')
            res = subprocess.run([
                sys.executable, '-m', 'pip', 'install', '--no-index',
                f'--find-links={wd}', 'arc-agi', 'arcengine'
            ], capture_output=True, text=True)
            if res.returncode == 0:
                print('[OK] arc-agi and arcengine installed from offline wheels.')
                installed = True
                break
            else:
                print(f'[WARN] Offline wheel installation warning: {res.stderr}')
    if not installed:
        raise RuntimeError('arc-agi and arcengine packages not found and offline wheels missing.')

# 2. Check GPU Accelerator & Memory
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available. GPU accelerator must be enabled in Notebook Settings.')
    else:
        free_b, total_b = torch.cuda.mem_get_info()
        gpu_name = torch.cuda.get_device_name(0)
        print(f'[OK] GPU Accelerator: {gpu_name}')
        print(f'[OK] Total VRAM:      {total_b / (1024**3):.2f} GB (Free: {free_b / (1024**3):.2f} GB)')
        print(f'[OK] bfloat16:        {"Supported" if torch.cuda.is_bf16_supported() else "Not Supported"}')
except ImportError:
    raise RuntimeError('PyTorch is not installed.')

# 3. Adapter Explicitly Disabled (Zero-Shot Base Mode)
adapter_path = None
adapter_bytes = 0
print('[INFO] Operating in Base-Model-Only Zero-Shot Mode (REAL_MODEL_ZERO_SHOT_BASE).')
print('       SAM-AI LoRA adapter is NOT loaded in this evaluation.')

# 4. Check Official ARC-3 Environment Data via Arcade
found_envs = len(list(Path(ENV_DATA_DIR).rglob('metadata.json')))
print(f'[OK] Found {found_envs} official ARC-3 environment files in {ENV_DATA_DIR}.')
print('\n[READY] All hardware, base model, and environment prerequisites verified.')


In [ ]:
# ==============================================================================
# 2. Sovereign ModelRunner Loading (Base Model Only)
# ==============================================================================
from sam_ai.substrate.models import HuggingFaceModelRunner

print(f'[*] Initializing sovereign HuggingFaceModelRunner with base model...')
runner = HuggingFaceModelRunner(
    model_name_or_path=base_model_path,
    adapter_path=None, # Strictly base-model zero-shot
    device_map='auto',
    load_in_4bit=True,
)
runner.load_model()

is_adapter_active = runner.is_adapter_loaded
assert not is_adapter_active, 'CRITICAL: Adapter must not be loaded in zero-shot base evaluation.'
print(f'[OK] ModelRunner ready.')
print(f'     Model Name:     {runner.model_name}')
print(f'     Adapter Active: {is_adapter_active} (Zero-Shot Base Mode)')


In [ ]:
# ==============================================================================
# 3. SAM-AI Agent Setup (Single-Variable Isolation: Arm B + Smoke Sampling)
# ==============================================================================
from sam_ai.agent.arc3_agent import SAMARC3Agent

# Initialize SAM-AI Agent with smoke-test sampling and hygiene
agent = SAMARC3Agent(
    model_runner=runner,
    enable_verification_guard=True,
    enable_failure_memory=False, # Single-variable isolation
    is_real_model=True,
    temperature=0.6,
    top_p=0.95,
    seed=42,
    max_tokens=3072,
    use_system_prompt=False,
    stop_on_parse_failure=True,
)
print('[OK] SAM-AI Agent initialized under Arm B single-variable isolation.')


In [ ]:
# ==============================================================================
# 4. Official Arcade Execution (Strictly Max 5 Steps) & Telemetry
# ==============================================================================
from arc_agi import Arcade, OperationMode

arcade = Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENV_DATA_DIR,
)
available_envs = arcade.get_environments()
if not available_envs:
    raise RuntimeError(f'No environments found in {ENV_DATA_DIR}')

chosen_game = available_envs[0].game_id
print(f'[*] Initializing official environment for game: {chosen_game}')
env = arcade.make(game_id=chosen_game, seed=0)
if env is None:
    raise RuntimeError(f'arcade.make returned None for game {chosen_game}')

print(f'[*] Launching interactive smoke session (strictly max 5 steps)...')
t0 = time.perf_counter()
summary = agent.run_session(env, game_id=chosen_game, max_steps=5)
elapsed = time.perf_counter() - t0

# Persist Telemetry (Strictly REAL_MODEL_ZERO_SHOT_BASE)
out_dir = Path('evidence/real_model_smoke_test')
out_dir.mkdir(parents=True, exist_ok=True)

eval_type = 'REAL_MODEL_ZERO_SHOT_BASE'
report = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'game_id': chosen_game,
    'evaluation_type': eval_type,
    'is_real_model': True,
    'adapter_loaded': False,
    'adapter_path': None,
    'adapter_bytes': 0,
    'base_model': base_model_path,
    'steps_taken': summary.total_steps,
    'levels_completed': summary.levels_completed,
    'final_state': summary.final_state,
    'success': summary.success,
    'verification_failures': summary.verification_failures,
    'elapsed_seconds': elapsed,
    'steps_telemetry': [
        step.telemetry.to_dict() for step in summary.steps_history if step.telemetry is not None
    ],
    'action_history': [
        {'step': s.step_idx, 'action': s.executed_action, 'status': s.verification_status.value}
        for s in summary.steps_history
    ],
}

with open(out_dir / 'smoke_summary.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
with open('smoke_summary.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

with open(out_dir / 'smoke_trace.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(report) + '\n')
with open('smoke_trace.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(report) + '\n')

print('=' * 75)
print(f'SMOKE TEST COMPLETED: {chosen_game} | Steps: {summary.total_steps} | State: {summary.final_state}')
print(f'Evaluation Type: {eval_type}')
print(f'Summary file:    {out_dir / "smoke_summary.json"}')
print('=' * 75)
